## Reference Data

In [ ]:
%pip uninstall langchain langchain-community langchain-core -y

In [ ]:
%pip install \
langchain==0.3.25 \
langchain-community==0.3.24 \
langchain-core==0.3.66

In [ ]:
%pip install ragas

In [ ]:

reference_data = [
  {
    "question": "What’s the leave policy?", #Office condos for sale in 94539
    "ground_truth": "employees must submit a leave request for approval.", #Expected llm context
    #"context": "Employees must submit a leave request for approval. " #Expected context from source documents based on the question #2 properties
  }
]
question = reference_data[0]['question']
ground_truth = reference_data[0]['ground_truth']
context = ground_truth #reference_data[0]['context']
print (f"question: {question}")
print (f"ground_truth: {ground_truth}")
print (f"context: {context}")

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from sentence_transformers import SentenceTransformer

from pymilvus import (
    connections,
    Collection,
    CollectionSchema,
    FieldSchema,
    DataType,
    utility,
)
load_dotenv(override=True, dotenv_path="../.env.local")

milvus_uri = os.getenv("MILVUS_URI")
milvus_token = os.getenv("MILVUS_API_KEY")


# Retrieve context from Milvus DB
def retrieve_similiar_contexts(query, collection_name="employee_policies", top_k=1):
    """
    Given a user query, return top K semantically similar texts from Milvus.
    """
    # connections.connect("default", host="localhost", port="19530")
    connections.connect(
        alias="default",
        uri=milvus_uri,
        token=milvus_token
    )

    print("Connected to Milvus on Zilliz Cloud")

    collection = Collection(collection_name)
    model = SentenceTransformer("all-MiniLM-L6-v2")

    query_vector = model.encode([query]).tolist()

    results = collection.search(
        data=query_vector,
        anns_field="embedding",
        param={"metric_type": "COSINE", "params": {"nprobe": 10}},
        limit=top_k,
        output_fields=["content"]
    )
    print(f"Results: {results}")
    top_docs = []
    for hit in results[0]:
        top_docs.append({
            "content": hit.entity.get("content"),
            "score": hit.distance
        })

    return top_docs



In [ ]:


#from milvus_chatbot_with_rag import retrieve_similiar_contexts

def perform_retrieval(question):
    collection_name = "policy_docs_collection_2"
    retrieved_context = retrieve_similiar_contexts(question, collection_name, 3)[0]['content']
    print (f"perform_retrieval.retrieved_context: {retrieved_context}")
    return retrieved_context

    
# retrieved_context = perform_retrieval("What’s the leave policy?")
retrieved_context = perform_retrieval(question)

In [ ]:

# Validate retrieved context against reference data    

print(f"Retrieved context: {retrieved_context}")
print(f"Expected context: {context}")
print(f"question: {question}")

In [ ]:
%pip install datasets 

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import context_precision, context_recall

from dotenv import load_dotenv
from openai import OpenAI
import os

# --- Load API Key ---
load_dotenv(override=True, dotenv_path="../.env.local")
my_api_key = os.getenv("OPENAI_API_KEY")


client = OpenAI(api_key=my_api_key)

# question = reference_data[0]['question']
# ground_truth = reference_data[0]['ground_truth']
# context = reference_data[0]['context']
# Question User asked
question = reference_data[0]['question']

# Reference context (should be a string)
reference_context = reference_data[0]['ground_truth']

# Retrieved context (a string from perform_retrieval)
retrieved_context = [perform_retrieval(question)]

# Build dataset properly
dataset_dict = {
    "question": [question],
    "ground_truth": [reference_context],   # single string
    "contexts": [retrieved_context],      # list of strings INSIDE another list
    "answer": [""]
}

print(f"dataset_dict: {dataset_dict}")

ragas_dataset = Dataset.from_dict(dataset_dict)


# Evaluate retrieval
results = evaluate(
    dataset=ragas_dataset,
    metrics=[context_precision, context_recall]
)


In [ ]:
print("Retrieval Evaluation Results:")
results.to_pandas()


## 